# 3장 2강: Chain Rule과 Backpropagation — 실습문제

## 실습 목표

- 연결된 함수에서 중간값을 먼저 계산합니다.
- 주어진 미분 규칙으로 각 단계의 변화율을 구하고 곱합니다.
- 순전파의 계산값과 역전파의 기울기를 구별합니다.
- 교안의 작은 회귀·분류 모델에서 가중치와 편향의 기울기를 구합니다.
- 역전파와 파라미터를 실제로 바꾸는 과정이 다름을 설명합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용하며 외부 데이터 파일은 필요하지 않습니다. 제공된 「3장 2강: Chain Rule과 Backpropagation」 본문과 연습문제의 식·숫자를 사용합니다. 실제 구매는 y=1, 미구매는 y=0으로 표시합니다.

## 실습 구성과 교안 대응

| 구분 | 실습 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | f=(3x+1)², x=2의 두 단계 Chain Rule | 1절 본문: 합성함수와 특정 지점의 기울기 |
| 필수 1 / 문제 1-2 | 두 단계 회귀 모델의 순전파·역전파 | 2절 본문: x=2, y=12, w₁=2, b₁=1, w₂=2, b₂=0 |
| 필수 2 / 문제 2-1 | 실제 구매 고객의 Sigmoid+BCE 역전파 | 3절 본문: x=2, y=1, w=0.5, b=0 |
| 필수 2 / 문제 2-2 | 제공된 순서표로 ReLU 은닉층까지 연결 | 3절 본문: x=2, y=1, w₁=1, b₁=0, w₂=0.5, b₂=0 |
| 과제 / 문제 3-1 | f=(2x+3)², x=1의 기본 미분 복습 | 연습문제 1의 문제 1·2 |
| 과제 / 문제 3-2 | 실제 미구매 고객의 기본 Gradient 복습 | 연습문제 3의 문제 1~3 중 기본 계산 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 기초 실습 4문제와 복습 과제 2문제로 재구성했습니다. 새로운 미분법이나 교안에 없는 학습 알고리즘은 사용하지 않습니다.

## 진행 안내

1. 실습 준비 셀을 실행해 NumPy를 불러옵니다.
2. 문제에 제시된 식을 읽고 입력값부터 순서대로 계산합니다.
3. 제공된 미분식과 계산 순서표에 숫자를 대입합니다. 공식을 외우거나 혼자 유도할 필요는 없습니다.
4. 기울기를 계산할 때에는 **전달받은 손실 기울기 × 현재 연산의 미분값**을 구합니다.
5. 문제 2-2는 연결 단계가 많으므로 제공 표를 한 줄씩 따라가세요. 모든 단계를 한 번에 외울 필요는 없습니다.
6. 소수는 계산 중간에 반올림하지 않고 출력할 때만 소수점 아래 세 자리로 반올림합니다.
7. Q1~Q4에 답하세요. 숫자뿐 아니라 무엇을 계산한 값인지 함께 적습니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 함수 / 함수값 | 입력으로 출력을 계산하는 규칙 / 그 규칙에 숫자를 넣은 결과 |
| 합성함수 | 한 함수의 출력이 다음 함수의 입력으로 들어가는 구조 |
| 중간 변수 z, a, h | 연산 사이에서 결과를 보관하는 이름. 정확한 뜻은 각 문제의 식에서 확인 |
| 미분값·기울기 | 현재 점에서 입력을 아주 조금 바꿀 때 출력이 변하는 방향과 정도 |
| 국소 미분값 | 전체 계산 중 한 연산만 따로 본 변화율. ‘국소’는 여기서 ‘한 단계’라는 뜻 |
| Chain Rule / 연쇄법칙 | 연결된 단계의 미분값을 곱해 전체 변화율을 구하는 규칙 |
| 계산 그래프 | 어떤 값이 어떤 연산을 거쳐 다음 값에 연결되는지 나타낸 구조 |
| 순전파 | 입력부터 중간값·예측·손실을 순서대로 계산하는 과정 |
| 역전파 / Backpropagation | 최종 손실부터 이전 연산으로 기울기를 전달하는 계산 과정 |
| x / y | 입력 데이터 / 실제 정답. 문제에서 주어진 고정값 |
| w / b | 입력에 곱하는 가중치 / 더하는 편향. 모델에서 조절하는 수인 파라미터 |
| w₁, b₁ / w₂, b₂ | 첫 번째 / 두 번째 연산의 파라미터. 아래 숫자는 단계 표시이며 제곱이 아님 |
| ŷ / y_pred | 모델의 회귀 예측값. ŷ는 ‘와이 햇’으로 읽음 |
| L / loss | 예측을 정답과 비교해 계산한 손실 |
| Logit z | Sigmoid로 확률을 만들기 전 점수. z 자체를 확률로 읽지 않음 |
| Sigmoid / p | 점수를 0과 1 사이 확률로 바꾸는 함수 / 이 자료에서는 구매 확률 |
| BCE | 구매·미구매처럼 두 종류를 예측할 때 사용하는 손실 |
| ReLU | 입력이 양수이면 그대로, 음수이면 0을 내보내는 함수 |
| 은닉층 / h | 입력과 최종 출력 사이의 중간 처리 단계 / 이 자료에서는 ReLU를 거친 중간값 |
| Gradient / ∇L | 파라미터별 손실 기울기를 정해진 순서대로 묶은 것 |
| 파라미터 업데이트 | 계산한 기울기를 이용해 w, b 등의 값을 실제로 변경하는 일 |
| ≈ / ln | 대략 같음 / 자연로그 |

### 미분 기호를 읽는 방법

`df/dz`는 **z에 대한 f의 순간 변화율**입니다. 분수처럼 보여도 f의 값에서 z의 값을 단순히 나누라는 뜻은 아닙니다. `∂L/∂w`는 다른 독립적인 변수는 고정하고 w를 바꿀 때의 손실 변화율입니다. w의 변화가 중간값을 거쳐 L까지 전달되는 영향은 포함합니다.

코드의 `dL_dw`는 `∂L/∂w`를 저장하는 이름입니다. `dL_dz`는 z에 대한 손실 기울기이고 `dp_dz`는 z에 대한 확률의 변화율입니다. **앞의 문자와 뒤의 문자를 모두 읽어야 무엇의 변화율인지 알 수 있습니다.**

### 사용할 기본 미분 규칙

| 연산 | 미분 대상 | 미분한 결과 |
|---|---|---|
| z=3x+1 | x | 3 |
| f=z² | z | 2z |
| ŷ=wx+b | w / b | x / 1 |
| L=(y−ŷ)² | ŷ | 2(ŷ−y), 실제 정답 y는 고정 |
| p=Sigmoid(z) | z | p(1−p) |
| Sigmoid와 BCE를 연결한 L | z | p−y |

함수값을 계산하는 식과 미분값을 계산하는 식을 구분하세요. 예를 들어 f=z²의 출력은 z²이고, 그 연산의 미분값은 2z입니다.

**지수·로그가 처음인 경우:** e는 약 2.718인 수이고 `np.exp(t)`는 e의 t제곱을 계산합니다. 자연로그 ln은 e를 몇 제곱하면 주어진 수가 되는지 나타내는 계산입니다. 이번 실습에서는 지수·로그 값을 손으로 구하지 않고 제공된 코드를 사용합니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 계산을 돕는 도구이고, `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 과제. 기본 계산 복습

### 문제 3-1. 두 단계 Chain Rule 복습하기

#### 문제 설명

교안 연습문제 1의 함수 $f(x)=(2x+3)^2$에서 x=1을 사용합니다. 문제 1-1과 같은 방법으로 풉니다.

$$z=2x+3,\qquad f=z^2$$

#### 요구사항

1. z와 f를 차례로 계산하세요.
2. df/dz=2z, dz/dx=2를 계산하고 곱해 df/dx를 구하세요.
3. 계산값을 출력하고 Q1~Q4에 답하세요.

#### 작성 도움말

문제 1-1의 코드를 참고하되 안쪽 식을 `z = 2 * x + 3`, 입력을 `x = 1.0`으로 바꾸세요. 안쪽 미분값 `dz_dx`도 이번 식에 맞추어 2로 바꿉니다. 과제에서는 합성함수 하나만 계산하며 추가 회귀 모델이나 세 단계 함수 계산은 요구하지 않습니다.

#### 해석 질문

**Q1.** 중간값 z와 함수값 f는 각각 얼마인가요?

**Q2.** 두 국소 미분값 df/dz와 dz/dx는 각각 얼마인가요?

**Q3.** x=1에서의 전체 미분값 df/dx는 얼마인가요?

**Q4.** 전체 미분값이 양수라는 것은 무엇을 뜻하나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-1. 두 단계 Chain Rule 복습하기
# 1. 제공된 숫자와 코드를 참고해 순전파를 계산하세요.
# 2. 미분식 또는 순서표에 따라 기울기를 계산하세요.
# 3. 결과를 출력하고 아래 질문에 답하세요.

In [ ]:
x = 1.0
z = 2 * x + 3
f = z ** 2
df_dz = 2 * z
dz_dx = 2.0
df_dx = df_dz * dz_dx

In [ ]:
print("z: ", z)
print("f: ", f)
print("df_dz: ", df_dz)
print("dz_dx: ", dz_dx)
print("df_dx: ", df_dx)

z:  5.0
f:  25.0
df_dz:  10.0
dz_dx:  2.0
df_dx:  20.0


#### 답변 작성란

**Q1.** 중간값 z와 함수값 f는 각각 얼마인가요?

답변: 
```python
z:  5.0
f:  25.0
```

**Q2.** 두 국소 미분값 df/dz와 dz/dx는 각각 얼마인가요?

답변: 
```python
df_dz:  10.0
dz_dx:  2.0
```

**Q3.** x=1에서의 전체 미분값 df/dx는 얼마인가요?

답변: 
```python
df_dx:  20.0
```

**Q4.** 전체 미분값이 양수라는 것은 무엇을 뜻하나요?

답변: x에서 x축의 양의 방향으로 조금 이동할 때 f값도 증가하는 방향으로 이동한다.

### 문제 3-2. 미구매 고객의 기본 Gradient 복습하기

#### 문제 설명

교안 연습문제 3의 미구매 고객입니다. 이번 실제 정답은 y=0입니다.

$$z=wx+b,\qquad p=\operatorname{Sigmoid}(z),\qquad L=\operatorname{BCE}(y,p)$$

| x | y | w | b |
|---:|---:|---:|---:|
| 3 | 0 | 0.2 | 0.1 |

p는 여전히 **구매 확률**이고, 미구매 확률은 1−p입니다. 실제 정답이 0이라고 해서 p의 뜻이 미구매 확률로 바뀌지 않습니다.

#### 요구사항

1. 문제 2-1의 제공 함수로 z, p, 1−p, BCE 손실을 계산하세요.
2. 교안의 정리된 식 `dL_dz = p - y`를 사용하세요.
3. `dL_dw = dL_dz * x`, `dL_db = dL_dz * 1`을 계산해 [w,b] 순서로 출력하세요.
4. Q1~Q4에 답하세요. 출력은 소수점 아래 세 자리로 반올림합니다.

#### 작성 도움말

BCE 코드는 `loss = np.logaddexp(0, z) - y * z`입니다. y=0인 경우 손실의 의미는 −ln(1−p)입니다. 이번 과제는 제공된 p−y 공식을 사용하며 BCE와 Sigmoid의 일반 미분식을 다시 유도하거나 별도로 검증할 필요는 없습니다.

#### 해석 질문

**Q1.** z, 구매 확률 p, 미구매 확률 1−p는 각각 얼마인가요?

**Q2.** BCE는 얼마이며 왜 −ln(1−p)를 사용하나요?

**Q3.** 점수 기울기와 [w,b] 순서의 Gradient는 무엇인가요?

**Q4.** 현재 손실을 줄이려면 각 파라미터를 어느 쪽으로 조금 바꾸나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-2. 미구매 고객의 기본 Gradient 복습하기
# 1. 제공된 숫자와 코드를 참고해 순전파를 계산하세요.
# 2. 미분식 또는 순서표에 따라 기울기를 계산하세요.
# 3. 결과를 출력하고 아래 질문에 답하세요.

In [ ]:
x= 3
y= 0
w= 0.2
b= 0.1

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = w * x + b
p = sigmoid(z)
loss = np.logaddexp(0, z) - y * z
# 교안에서 두 미분값의 곱은 p−y로도 정리됩니다. 긴 식을 유도할 필요는 없습니다. 
# 함수와 손실 코드는 그대로 사용하세요. 
# 출력만 소수점 아래 세 자리로 반올림합니다.

In [ ]:
dL_dz= p - y
dL_dw= dL_dz * x
dL_db= dL_dz * 1

print("z:", round(z, 3))
print("p:", round(p, 3))
print("1 - p:", round(1 - p, 3))
print("BCE:", round(loss, 3))

print("dL_dz:", round(dL_dz, 3))
print("Gradient [w, b]:", [round(dL_dw, 3), round(dL_db, 3)])

z: 0.7
p: 0.668
1 - p: 0.332
BCE: 1.103
dL_dz: 0.668
Gradient [w, b]: [np.float64(2.005), np.float64(0.668)]


#### 답변 작성란

**Q1.** z, 구매 확률 p, 미구매 확률 1−p는 각각 얼마인가요?

답변: 
```python
z: 0.7
p: 0.668
1 - p: 0.332
```

**Q2.** BCE는 얼마이며 왜 −ln(1−p)를 사용하나요?

답변: 
```python
BCE: 1.103
```
실제 정답이 y=0(미구매)이면 BCE 식 −[y·ln p + (1−y)·ln(1−p)]에서 y·ln p 항이 사라지고 −ln(1−p)만 남습니다. 이때 1−p는 모델이 **실제 정답(미구매)에 부여한 확률**, 즉 이 고객 데이터의 우도입니다. 우도는 클수록 좋은 값인데, 최대값을 구하는 것은 일반적인 손실 함수의 최솟값을 구하는 것과 통일되지 않기 때문에 음의 부호를 붙여준다. 자연로그를 취해주는 이유는 확률값끼리 곱해지다 보면 값이 너무 작아져 부동 소수점에 의해 0으로 처리될 수 있기 때문에 로그를 취해서 적절한 값으로 취환해주는 것이다.

**Q3.** 점수 기울기와 [w,b] 순서의 Gradient는 무엇인가요?

답변: 
```python
dL_dz: 0.668
Gradient [w, b]: [2.005, 0.668]
```

**Q4.** 현재 손실을 줄이려면 각 파라미터를 어느 쪽으로 조금 바꾸나요?

답변: w와 b 둘다 음의 방향으로 이동하면 된다.

---

## 실습 마무리

다음 질문으로 핵심 내용을 복습하세요.

**1.** Chain Rule에서는 연결된 단계의 미분값을 어떻게 사용하나요?

**2.** 순전파와 역전파는 각각 무엇을 구하나요?

**3.** L=(y−ŷ)²의 예측값 기울기에서 −1을 곱하는 이유는 무엇인가요?

**4.** Sigmoid와 BCE를 연결했을 때 p−y는 무엇인가요?

**5.** ReLU 입력 a=2일 때 출력과 미분값은 각각 얼마인가요?

**6.** 역전파와 파라미터 업데이트는 같은 과정인가요?

### 실습 마무리 답변 작성란

**1.** 답변: 역전파는 최종 손실에서 출발해 거꾸로 이동하면서, **전달받은 손실 기울기에 현재 연산의 국소 미분값을 곱해** 다음 기울기를 구합니다. 이 곱셈을 단계마다 이어 가면 각 중간값과 파라미터(w, b)의 기울기를 얻습니다. 예를 들어 `dL_dw2 = dL_dy_pred * a`, `dL_da = dL_dy_pred * w2`처럼 계산합니다.

**2.** 답변: 순전파는 입력부터 시작해 중간값, 예측값, 손실을 순서대로 **계산값**으로 구하는 과정입니다. 역전파는 최종 손실에서 거꾸로 이동하며 각 중간값과 파라미터에 대한 손실의 **기울기(Gradient)** 를 구하는 과정입니다.

**3.** 답변: L=(y−ŷ)²을 ŷ로 미분할 때 바깥 함수 (·)²의 미분값 2(y−ŷ)에 안쪽 식 (y−ŷ)를 ŷ로 미분한 값 **−1**이 Chain Rule로 곱해지기 때문입니다.

**4.** 답변: p−y는 Sigmoid와 BCE를 연결했을 때 **점수 z에 대한 손실 기울기 ∂L/∂z**입니다.

Chain Rule에 따라 ∂L/∂z = (∂L/∂p) × (dp/dz)입니다. BCE를 p로 미분하면 ∂L/∂p = −y/p + (1−y)/(1−p) = (p−y) / [p(1−p)]이고, Sigmoid의 미분은 dp/dz = p(1−p)입니다. 두 값을 곱하면 p(1−p)가 약분되어 ∂L/∂z = **p − y**만 남습니다.

예를 들어 y=1이면 (−1/p)×p(1−p) = p−1이고, y=0이면 (1/(1−p))×p(1−p) = p입니다. 3-2 문제(p≈0.668, y=0)에서도 3.014 × 0.222 ≈ 0.668로 p−y와 일치합니다.

즉 기울기의 크기는 예측 확률 p와 실제 정답 y의 차이이고, 부호는 z를 어느 쪽으로 바꾸면 손실이 늘어나는지를 알려줍니다.

**5.** 답변: a > 0 이므로 출력은 입력값과 같은 2이고, 미분값은 1이다.

**6.** 답변: 다른 과정입니다. 역전파는 현재 파라미터에서 손실의 기울기(Gradient)를 계산하는 과정이고
- 파라미터 업데이트는 학습률과 negative gradient를 이용하여 파라미터들을 조정하는 과정이다.